# Pre-process eQTL meta-analysis data

This script reads in the eQTL meta-analysis output at [syn16984815](https://www.synapse.org/#!Synapse:syn16984815) and condenses it for ingest into Agora. For each gene (Ensembl ID), there are multiple SNPs with different p-values / FDRs. Here we take the smallest FDR for each gene, mark that gene as significant if the smallest FDR is <= 0.05, and discard all other duplicate gene rows. 

The output is a data frame with one row per Ensembl ID, with two columns: one for the Ensembl ID and one for whether the gene was significant for at least one SNP. The output file is uploaded to [syn12514912](https://www.synapse.org/#!Synapse:syn12514912).

In [ ]:
from agoradatatools.etl import utils
import pandas as pd
from synapseclient.models import File

syn = utils._login_to_synapse()

This file will take awhile to download, as it's >17 GB. 

In [ ]:
eqtl_meta = await File(id="syn16984815").get_async()

This file is extremely large and has a lot of columns we don't need, so we only have `read_csv` return the two columns we are interested in. 

In [ ]:
eqtl_meta = pd.read_csv(eqtl_meta.path, usecols=["gene", "FDR"])
eqtl_meta.head()

Take the minimum FDR for each gene, and determine significance. 

In [ ]:
eqtl_meta = eqtl_meta.groupby("gene")["FDR"].agg("min").reset_index()
eqtl_meta["has_eqtl"] = eqtl_meta["FDR"] <= 0.05
eqtl_meta["has_eqtl"].value_counts()

Rename "gene" column and get rid of "FDR" column.

In [ ]:
eqtl_meta = eqtl_meta.rename(columns = {"gene": "ensembl_gene_id"}).drop(columns="FDR")
eqtl_meta

Make sure there are no NA values in the data frame.

In [ ]:
print(any(eqtl_meta["ensembl_gene_id"].isna()))
print(any(eqtl_meta["has_eqtl"].isna()))

Write to a file.

In [ ]:
eqtl_meta.to_csv("../output/eqtl_meta_analysis.csv", index=False)